# Modelos direto por UF · 4/4 · escolha (2018–2021) → teste (2022–2025)

**Antes de rodar (uma vez só)**, em `MyDrive/dengue_V4/` (que já tem `dados/`, `referencias_imdc3/` e os arquivos da competição):
1. suba **`uf_lib.py`** e **`uf_autoarima.R`** (pasta `data_imdc_2026/UF/`);
2. suba a **pasta `UF/dados/` com o nome `dados_uf/`** (5 arquivos `.parquet` + `vizinhanca_uf.npz`);
3. substitua **`comp_modelos.py`** e **`comp_inla.R`** pelas versões novas (de `COMPETICAO/`): elas trazem o INLA não linear e a correção do `val_size`.

Os resultados vão para `MyDrive/dengue_V4/resultados_uf/`. Se a sessão cair, rode de novo: o que já foi feito é pulado.
⚠️ Ao abrir vários notebooks ao mesmo tempo, **espere a célula 1 do primeiro terminar** antes de rodar os outros: sessões simultâneas podem criar duas pastas `resultados_uf` no Drive. Se isso acontecer, renomeie a cópia para `resultados_uf_2`; o `CompUF4` e o `Comp5` leem todas as pastas `resultados_uf*`.

**Temporadas:** 2018–2021 = **afinação** (é onde se escolhe tudo); 2022–2025 = **teste** (S1–S4, olhado uma vez no `CompUF4`).

| Notebook | O que roda | Ambiente | Tempo estimado |
|---|---|---|---|
| `CompUF1_portados` | os 13 modelos da competição, direto por UF | CPU High-RAM | 1–2 h (com INLA) |
| `CompUF2_CPU` | auto.arima / auto.ARIMAX (Python e R), 6 VARs, INLA não linear, dois estágios, GBMs com covariáveis, ablação | CPU High-RAM | 4–6 h |
| `CompUF3_GPU` | AGCRN, Chronos, 6 redes com pooling (com e sem exógenas), STID, 4 redes de grafo, Chronos-2 | GPU L4 | 8–12 h |
| `CompUF4_avaliacao` | escolha em 2018–2021, teste em 2022–2025, ensembles | CPU | ~15 min |

**Protocolo (definido antes de olhar o teste):**
1. Tudo é escolhido nas temporadas de **afinação (2018–2021)**, com a régua **R vs climatologia** (o BB oficial só existe para 2022–2025).
2. Entram na escolha só modelos com as 4 temporadas de afinação.
3. Para cada um dos 15 melhores, escolhe-se a **largura dos intervalos** (λ) que minimiza o R na afinação.
4. Candidatos finais: o melhor modelo sozinho e a **mediana dos quantis dos k melhores** (k = 3, 5, 7), misturando "UF direto" e "região→UF".
5. **O final é o de menor R na afinação.** Só depois disso o teste (2022–2025, contra o BB e as 28 equipes do 3º IMDC) é calculado.

Lê `resultados_uf/` (UF direto) e `resultados_competicao/` (região→UF).

In [ ]:
NOTEBOOK = 'uf_avaliacao'
PACOTES = ''

# @title Drive, arquivos e pacotes
import os, sys, shutil, time, glob, subprocess, json, traceback
import numpy as np, pandas as pd
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PASTA_DRIVE, PASTA_LOCAL, NO_COLAB = '/content/drive/MyDrive/dengue_V4', '/content/uf', True
except ImportError:
    PASTA_DRIVE = PASTA_LOCAL = os.path.abspath('.'); NO_COLAB = False
if NO_COLAB:
    os.makedirs(PASTA_LOCAL, exist_ok=True)
    for item in ('v4_lib.py', 'comp_lib.py', 'comp_modelos.py', 'comp_inla.R', 'uf_lib.py', 'uf_autoarima.R', 'dados',
                 'dados_uf', 'referencias_imdc3'):
        o, d = os.path.join(PASTA_DRIVE, item), os.path.join(PASTA_LOCAL, item)
        assert os.path.exists(o), f'Faltou subir {item} para {PASTA_DRIVE}'
        shutil.copytree(o, d, dirs_exist_ok=True) if os.path.isdir(o) else shutil.copy(o, d)
    if PACOTES:
        r = subprocess.run(f'pip install -q {PACOTES}', shell=True, capture_output=True, text=True)
        print(r.stdout[-400:], r.stderr[-400:])
sys.path.insert(0, PASTA_LOCAL)
PASTA_RES = os.path.join(PASTA_DRIVE, 'resultados_uf')
PASTA_COMP = os.path.join(PASTA_DRIVE, 'resultados_competicao')
os.makedirs(os.path.join(PASTA_RES, 'graficos'), exist_ok=True)
import importlib, v4_lib as V, comp_lib as L, comp_modelos as CM, uf_lib as U
for m in (V, L, CM, U):
    importlib.reload(m)
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'nenhuma', '· CPUs:', os.cpu_count())
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 40)
ctx = U.ContextoUF(os.path.join(PASTA_LOCAL, 'dados'), os.path.join(PASTA_LOCAL, 'referencias_imdc3'),
                   os.path.join(PASTA_LOCAL, 'dados_uf'))
TUNING, TESTE, TODAS = U.TUNING, U.TESTE, U.TODAS_UF
M = 500                                  # trajetórias por UF
REFAZER = False

## 1. Previsões disponíveis

In [ ]:
# Modelos de FORA (outra equipe, outro notebook, um CSV no formato da plataforma: date, adm_1, pred, lower_*, upper_*):
# liste aqui {nome: arquivo}. Eles são checados, gravados em resultados_externos/ e entram em tudo como os nossos.
EXTERNOS = {}            # ex.: {'EquipeX': os.path.join(PASTA_DRIVE, 'externos', 'equipex.csv')}
PASTA_EXT = os.path.join(PASTA_DRIVE, 'resultados_externos')
for nome, arq in EXTERNOS.items():
    print(nome, '→', L.importar_externo(arq, nome, PASTA_EXT, ctx.Du.ufs).to_dict('records') or 'ok')
# Todas as pastas resultados_uf* (sessões abertas ao mesmo tempo podem criar cópias com o mesmo nome no Drive;
# renomeie a cópia para resultados_uf_2, _3 … e ela entra aqui).
PASTAS_UF = [p for p in sorted(glob.glob(os.path.join(PASTA_DRIVE, 'resultados_uf*'))) if os.path.isdir(p)]
print('Pastas de resultados por UF:', [os.path.basename(p) for p in PASTAS_UF])
# Leitura padronizada: duplicatas → fica a fonte listada por último; previsões incompletas ficam fora (relatório abaixo)
FONTES = [('região→UF', PASTA_COMP)] + [(f'UF direto · {os.path.basename(p)}', p) for p in PASTAS_UF] + [('externo', PASTA_EXT)]
PU = L.carregar_previsoes(FONTES, ctx.Du.ufs, 'uf')
PU['fonte'] = PU.fonte.str.replace(r' · resultados_uf.*', '', regex=True)
PU = PU[~PU.modelo.str.startswith('ENS')]
cob = PU.groupby(['fonte', 'modelo']).season.agg(lambda s: ''.join('●' if a in set(s) else '·' for a in TODAS))
print('temporadas', TODAS[0], '→', TODAS[-1], '(● = existe)')
cob.to_frame('2018 … 2025')

## 2. Afinação (2018–2021): R vs climatologia por UF

In [ ]:
por_t, res_t = V.avaliar(PU[PU.season.isin(TUNING)], ctx.ver_uf, ref=('climatologia',))
fonte = PU.drop_duplicates('modelo').set_index('modelo').fonte
res_t['fonte'] = res_t.modelo.map(fonte)
res_t.to_csv(os.path.join(PASTA_RES, 'afinacao_ranking.csv'), index=False)
ELEGIVEIS = res_t[res_t.temporadas == len(TUNING)].dropna(subset=['R_vs_climatologia']).sort_values('R_vs_climatologia')
print(len(ELEGIVEIS), f'modelos com as {len(TUNING)} temporadas de afinação completas')
assert len(ELEGIVEIS), 'nenhum modelo com as 4 temporadas de afinação completas (veja o relatório da seção 1)'
ELEGIVEIS[['fonte', 'modelo', 'R_vs_climatologia', 'UFs_melhor_que_climatologia', 'dp_logR_vs_climatologia',
           'WIS_log_medio', 'IC50', 'IC90']].round(3).head(40)

## 3. UF direto × região→UF (mesmo modelo, afinação)

In [ ]:
r = res_t.set_index('modelo').R_vs_climatologia
pares = [(m[:-3], m) for m in r.index if m.endswith('_UF') and m[:-3] in r.index]
cmp_ = pd.DataFrame([(a, r[a], r[b]) for a, b in pares], columns=['modelo', 'região→UF', 'UF direto'])
cmp_['melhor'] = np.where(cmp_['UF direto'] < cmp_['região→UF'], 'UF direto', 'região→UF')
cmp_.round(3).sort_values('UF direto')

## 4. Ablação: quanto cada grupo de variáveis ajuda (GBM LightGBM, afinação)
> Razão > 1: tirar o grupo **piora** (o grupo ajuda). Razão < 1: tirar o grupo **melhora** (o grupo atrapalha).

In [ ]:
base = 'GBM_lgb_todas_UF'
if base in r.index:
    abl = pd.DataFrame([(g, r.get(f'GBM_lgb_todas_sem_{g}_UF', np.nan) / r[base]) for g in U.GRUPOS],
                       columns=['grupo retirado', 'R sem o grupo / R com todos'])
    display(abl.sort_values('R sem o grupo / R com todos', ascending=False).round(3))
else:
    print('Rode antes o CompUF2_CPU (modelo', base, ')')

## 5. Calibração da largura (λ escolhido na afinação)

In [ ]:
GRADE_LAM = [0.7, 0.8, 0.9, 1.0, 1.15, 1.3, 1.5, 1.75, 2.0]
TOP = ELEGIVEIS.modelo.head(15).tolist()
clim = PU[(PU.modelo == 'climatologia') & PU.season.isin(TUNING)]
lam = {}
for m in TOP:
    g = PU[(PU.modelo == m) & PU.season.isin(TUNING)]
    rs = []
    for l in GRADE_LAM:
        _, rr = V.avaliar(pd.concat([U.ajusta_largura(g, l).assign(modelo='x'), clim]), ctx.ver_uf, ref=('climatologia',))
        rs.append(float(rr.set_index('modelo').loc['x', 'R_vs_climatologia']))
    lam[m] = GRADE_LAM[int(np.argmin(rs))]
    print(f'{m:40s} λ = {lam[m]:4.2f} · R: {rs[GRADE_LAM.index(1.0)]:.3f} → {min(rs):.3f}')
CAL = pd.concat([U.ajusta_largura(PU[PU.modelo == m], l).assign(modelo=f'{m}+λ') for m, l in lam.items() if l != 1.0]
                + [PU.iloc[:0]], ignore_index=True)

## 6. Candidatos e escolha final (só com a afinação)

In [ ]:
TODOS = pd.concat([PU, CAL], ignore_index=True)
_, res_c = V.avaliar(pd.concat([TODOS[TODOS.season.isin(TUNING) & TODOS.modelo.isin(TOP + list(CAL.modelo.unique()))],
                               clim]), ctx.ver_uf, ref=('climatologia',))
res_c = res_c[res_c.modelo != 'climatologia']
melhor_versao = {}
for m in TOP:                                         # versão (com ou sem λ) de cada modelo
    v = [x for x in (m, f'{m}+λ') if x in set(res_c.modelo)]
    melhor_versao[m] = min(v, key=lambda x: float(res_c.set_index('modelo').loc[x, 'R_vs_climatologia']))
ordem = sorted(melhor_versao.values(), key=lambda x: float(res_c.set_index('modelo').loc[x, 'R_vs_climatologia']))
ENS = []
for k in sorted({min(k, len(ordem)) for k in (3, 5, 7)} - {1}):    # com poucos modelos, não repete o mesmo ensemble
    e, _ = V.ensemble(TODOS[TODOS.modelo.isin(ordem[:k])], ordem[:k], f'ENS_UF_top{k}', 'mediana')
    ENS.append(e)
ENS = pd.concat(ENS + [PU.iloc[:0][L.COLS_PREV]], ignore_index=True)
# grava os ensembles e as versões com λ: o Comp5 (negócio) e avaliações futuras leem daqui
pd.concat([ENS, CAL], ignore_index=True)[L.COLS_PREV].to_parquet(os.path.join(PASTA_RES, 'ensembles_uf.parquet'),
                                                                 index=False)
CAND = pd.concat([TODOS[TODOS.modelo == ordem[0]], ENS], ignore_index=True)
_, res_f = V.avaliar(pd.concat([CAND, clim])[lambda d: d.season.isin(TUNING)], ctx.ver_uf, ref=('climatologia',))
res_f = res_f[res_f.modelo != 'climatologia'].sort_values('R_vs_climatologia')
FINAL = res_f.iloc[0].modelo
print('Membros por ordem (afinação):', ordem[:7])
print('>>> ESCOLHA FINAL (antes de olhar o teste):', FINAL)
res_f[['modelo', 'temporadas', 'R_vs_climatologia', 'dp_logR_vs_climatologia', 'IC50', 'IC90']].round(3)

## 7. Teste (2022–2025): contra o BB oficial e as 28 equipes do 3º IMDC

In [ ]:
TESTE_DF = pd.concat([TODOS, ENS], ignore_index=True)
TESTE_DF = TESTE_DF[TESTE_DF.season.isin(TESTE)]
por_s, res_s = L.avaliar_uf(ctx, TESTE_DF, temporadas=TESTE)
res_s['destaque'] = np.where(res_s.modelo == FINAL, '◀ FINAL', np.where(res_s.modelo.str.startswith('IMDC:'), '3º IMDC', ''))
res_s = res_s[res_s.temporadas == len(TESTE)]
res_s.to_csv(os.path.join(PASTA_RES, 'teste_ranking.csv'), index=False)
res_s[['destaque', 'modelo', 'R_vs_BB', 'UFs_melhor_que_BB', 'dp_logR_vs_BB', 'R_vs_climatologia', 'IC50', 'IC90']].round(3).head(45)

In [ ]:
# @title Por temporada (o final e as referências)
ref = [FINAL, 'ARIMA_Fourier', 'climatologia', 'IMDC:emap_epidematicos_sarimax_state', 'IMDC:lncc_lncc_arp26_dengue']
t = por_s[por_s.modelo.isin(ref)]
b = por_s[por_s.modelo == 'BB'][['season', 'uf_code', 'WIS']].rename(columns={'WIS': 'WIS_BB'})
t = t.merge(b, on=['season', 'uf_code'])
t.assign(lr=np.log(t.WIS / t.WIS_BB)).groupby(['modelo', 'season']).lr.mean().pipe(np.exp).unstack().round(3)

In [ ]:
# @title Gravar a previsão final (para o Comp5 de negócio e para a submissão)
fin = pd.concat([TODOS, ENS])[lambda d: d.modelo == FINAL].copy()
fin.to_parquet(os.path.join(PASTA_RES, 'final_uf.parquet'), index=False)
arqs = V.exportar_submissao(fin, FINAL, os.path.join(PASTA_RES, 'submissao'))
print('final:', FINAL, '·', len(fin), 'linhas · arquivos de submissão:', arqs)
V.grafico_leque(pd.concat([fin, PU[PU.modelo.isin(['climatologia', 'ARIMA_Fourier'])]]), ctx.ver_uf,
                [FINAL, 'ARIMA_Fourier', 'climatologia'], TESTE[0],
                arquivo=os.path.join(PASTA_RES, 'graficos', f'leque_final_{TESTE[0]}.png'))